In [1]:
library("edgeR")
library("BiocParallel")
library('tidyverse')
library('data.table')
library(variancePartition)
library(RColorBrewer)
library(ggplot2)
library(corrplot)
library(pheatmap)
library(grid)

Warning message:
“package ‘edgeR’ was built under R version 4.3.3”
Loading required package: limma

Warning message:
“package ‘limma’ was built under R version 4.3.3”
Warning message:
“package ‘BiocParallel’ was built under R version 4.3.3”
── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.1     ✔ tibble    3.2.1
✔ lubridate 1.9.3     ✔ tidyr     1.3.1
✔ purrr     1.0.2     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors

Attaching package: ‘data.table’


The following objects are masked from ‘package:lubridate’:

    hour, isoweek, mday, minute, month, quarter, second, wday, week,
    yday, year


The following objects are masked from ‘package:dplyr’:

In [ ]:
expression <- fread('./Figures/data/allSamples_rsem_genes_results_mod.txt.gz') %>% as.data.frame() 
geneRef <- fread('./Figures/data/geneID2name.bed')
row.names(expression) <- expression$`V1`
expression$`V1` <- NULL
autosome_ChrX_expression = expression[rownames(expression) %in% geneRef[!geneRef$V1 %in% c('chrY') ]$V4,]
### Keep lncRNA and protein_coding
autosome_ChrX_expression = autosome_ChrX_expression[rownames(autosome_ChrX_expression) %in% geneRef[geneRef$V7 %in% c('lncRNA','protein_coding') ]$V4,]

In [ ]:
metadata <- fread('./Figures/data/PEC_RNAmeta.tsv.gz')  %>% as.data.frame() 
metadata <- metadata %>% replace_na(list(AGEYEARS = 0, AGEDAYS = 0))
metadata <- metadata %>% mutate(`Sample ID`= str_replace_all(`Sample ID`, "_", "-"))
metadata <- metadata %>% mutate(Age = AGEYEARS + AGEDAYS / 365)
qcdata <- fread('./Figures/data/metadata_RNA_QC.tsv.gz')  %>% as.data.frame() 

# DE analysis 

In [ ]:
### Choose celltype = GABA / GLU
Celltype  = 'GABA'
### Subset the sample that pass the QC
samples_to_keep <- qcdata$sample[qcdata['RNA_passQC'] == 'TRUE']
expression_passQC <- autosome_ChrX_expression %>% select(all_of(samples_to_keep))

Celltype_expression <- expression_passQC %>% select(matches(Celltype))

ID <- str_extract(names(Celltype_expression), paste(".*(?=_",Celltype,")",sep  = ''))
names(Celltype_expression) <- ID
Celltype_metadata <- metadata %>% 
  filter(`Sample ID` %in% ID, `Cell Type` == Celltype)
row.names(Celltype_metadata) <- Celltype_metadata$`Sample ID`
Celltype_metadata$`Sample ID` <- NULL
Celltype_metadata <-  Celltype_metadata[names(Celltype_expression),]

Celltype_metadata$period <- as.factor(Celltype_metadata$period)
Celltype_metadata$batch <- as.factor(Celltype_metadata$batch)
print(dim(Celltype_metadata))

[1] 54 18


In [ ]:
dge <- DGEList(Celltype_expression)
group = as.factor(Celltype_metadata$period)
dge$samples$group <- group
sex = as.factor( Celltype_metadata$sex)
dge$samples$ID <- row.names(dge$samples)
dge$samples$sex <- sex
dge$samples$Age <- Celltype_metadata$Age
dge$samples$batch <- as.factor(Celltype_metadata$batch)
dge$samples$race <- as.factor(Celltype_metadata$race)
dge$samples$`PMI Hours` <- (Celltype_metadata$`PMI Hours`)
dge$samples$RIN <- (Celltype_metadata$`RIN`)
dge <- calcNormFactors(dge, method = 'TMM')

In [ ]:
keep.rows <- rownames(dge$counts[apply((log2(cpm(dge,log=FALSE)+1) >= 3) , 1, sum) >= 3, ])
dge <- dge[keep.rows,, keep.lib.sizes=FALSE]
lcpm <- cpm(dge, log=TRUE)

## Make Contrast

In [ ]:
param <- SnowParam(12, "SOCK", progressbar = TRUE)
stage_order = make.names( c('periodinfancy', 'periodearlychildhood','periodlatechildhood','periodadolescence', 'periodadulthood', 'periodlateadulthood'))
form <- ~0 + period + batch + `PMI Hours`+ race + sex + RIN
comp_vec = c()
for (itr in 1:6){
    itr_nms2 <- stage_order[-c(1:itr)]
    itr_nms1 <- rep( c(stage_order[itr]), times=length(itr_nms2))
    comp_vec <- append( comp_vec, paste( itr_nms1, itr_nms2, sep='-'))
}
L <- makeContrastsDream( form, Celltype_metadata,contrasts=comp_vec)

In [ ]:
vobjDream <- voomWithDreamWeights(dge, form, Celltype_metadata, BPPARAM = param)
num_genes = dim(vobjDream$E)[1]

In [ ]:
fit <- dream(vobjDream, form, Celltype_metadata, L)
fit <- eBayes(fit)

In [ ]:
all_result = list()
for( i in 1:15){
    tmp <- topTable(fit, coef = i, number = num_genes)
    tmp$celltype <- Celltype
    tmp$comp <-  gsub("period", "", comp_vec[i])
    geneID <- rownames(tmp) 
    tmp <- cbind(geneID,tmp)
    rownames(tmp) <- NULL
    all_result[[i]] <- tmp
}
all_results <-  do.call(rbind, all_result)

In [ ]:
log2cpm <- log2(cpm(dge,log=FALSE)+1)

In [ ]:
filter_expression <- log2cpm[, order(Celltype_metadata$Age)]

In [ ]:
# inf_lateAdult = topTable(fit, coef = 5, number = num_genes)

In [ ]:
# write.csv(filter_expression,paste('/cndd/hex002/PsychEncode/rna_seq/',Celltype,'_DE_Dream.csv',sep = ''),row.names = TRUE)
# # write.csv(merged_df_Pvalues_filter,paste('/cndd/hex002/PsychEncode/rna_seq/',Celltype,'_DE_pvalue_table.csv',sep = ''),row.names = TRUE)
# # write.csv(max_FC_merged,paste('/cndd/hex002/PsychEncode/rna_seq/',Celltype,'_max_FC.csv',sep = ''),row.names = FALSE)
# # write.csv(merged_df_FC_filter,paste('/cndd/hex002/PsychEncode/rna_seq/',Celltype,'_FC.csv',sep = ''),row.names = TRUE)
# # write.csv(inf_lateAdult,paste('/cndd/hex002/PsychEncode/rna_seq/',Celltype,'_Infancy_LateAdulthood_FC.csv',sep = ''),row.names = TRUE)
# write.csv(all_results,paste('/cndd/hex002/PsychEncode/rna_seq/',Celltype,'_all_results.csv',sep = ''),row.names = FALSE)

## Combine results

In [ ]:
GABA_result = fread('/cndd/hex002/PsychEncode/rna_seq/GABA_all_results.csv')
GLU_result = fread('/cndd/hex002/PsychEncode/rna_seq/GLU_all_results.csv')
both_celltype = rbind(GABA_result,GLU_result)
write.csv(both_celltype,paste('/cndd/hex002/PsychEncode/rna_seq/Dream_all_results.csv',sep = ''),row.names = FALSE)